# Explore Bitcoin Fee Data

This notebook provides an exploratory data analysis (EDA) of the benchmark datasets.

## Contents
1. Load and inspect data
2. Feature distributions
3. Fee patterns over time
4. Correlations between features
5. Stable vs volatile periods
6. 3h vs 1d prediction challenges

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from benchmarks import (
    load_dataset,
    list_datasets,
    iter_snapshots,
    get_features,
    compute_inclusion_target,
    get_train_val_test_splits,
    FEATURE_COLUMNS,
    PREDICTION_HORIZONS,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('Setup complete!')

## 1. Load and Inspect Data

First, let's see what datasets are available and load one for exploration.

In [ ]:
# List available datasets
datasets = list_datasets()
print('Available datasets:')
for d in datasets:
    print(f'  - {d}')

In [ ]:
# Load the first dataset
dataset_name = datasets[0]
dataset_path = f'../data/local_datasets/{dataset_name}'
dataset = load_dataset(dataset_path)

print(f'Dataset: {dataset_name}')
print(f'Metadata:')
for k, v in dataset['metadata'].items():
    print(f'  {k}: {v}')

In [ ]:
# Check the index (list of snapshots)
index = dataset['index']
print(f'Total snapshots: {len(index)}')
print(f'\nSnapshot index sample:')
index.head()

In [ ]:
# Load a single snapshot to see the structure
for ts, snapshot in iter_snapshots(dataset):
    print(f'Snapshot at {ts}')
    print(f'Transactions: {len(snapshot)}')
    print(f'\nColumns: {snapshot.columns.tolist()}')
    print(f'\nFeature columns: {FEATURE_COLUMNS}')
    break

In [ ]:
# Sample of actual data
print('Sample transactions:')
snapshot[FEATURE_COLUMNS + ['first_seen_timestamp', 'block_timestamp']].head(10)

## 2. Feature Distributions

Let's examine the distribution of each feature.

In [ ]:
# Get features from a snapshot
features = get_features(snapshot)

# Basic statistics
print('Feature Statistics:')
features.describe()

In [ ]:
# Distribution plots for each feature
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLUMNS):
    ax = axes[i]
    data = features[col].dropna()
    
    # Use log scale for fee_rate and fee (often skewed)
    if col in ['fee_rate', 'fee', 'output_value']:
        data = data[data > 0]  # Filter out zeros for log
        ax.hist(np.log10(data), bins=50, edgecolor='black', alpha=0.7)
        ax.set_xlabel(f'log10({col})')
    else:
        ax.hist(data, bins=50, edgecolor='black', alpha=0.7)
        ax.set_xlabel(col)
    
    ax.set_ylabel('Frequency')
    ax.set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Fee rate distribution in detail (key feature for prediction)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1 = axes[0]
fee_rates = features['fee_rate'].dropna()
ax1.hist(fee_rates, bins=100, edgecolor='black', alpha=0.7)
ax1.axvline(fee_rates.median(), color='red', linestyle='--', label=f'Median: {fee_rates.median():.1f}')
ax1.axvline(fee_rates.mean(), color='orange', linestyle='--', label=f'Mean: {fee_rates.mean():.1f}')
ax1.set_xlabel('Fee Rate (sat/vbyte)')
ax1.set_ylabel('Frequency')
ax1.set_title('Fee Rate Distribution (Linear)')
ax1.legend()

# Log scale
ax2 = axes[1]
ax2.hist(np.log10(fee_rates[fee_rates > 0]), bins=100, edgecolor='black', alpha=0.7)
ax2.set_xlabel('log10(Fee Rate)')
ax2.set_ylabel('Frequency')
ax2.set_title('Fee Rate Distribution (Log Scale)')

plt.tight_layout()
plt.show()

# Percentiles
print('\nFee Rate Percentiles:')
for p in [5, 25, 50, 75, 90, 95, 99]:
    print(f'  {p}th: {fee_rates.quantile(p/100):.1f} sat/vbyte')

## 3. Fee Patterns Over Time

How do fees change throughout the dataset?

In [ ]:
# Collect fee statistics over time (sample every 10th snapshot for speed)
time_stats = []
for i, (ts, snapshot) in enumerate(iter_snapshots(dataset)):
    if i % 10 != 0:  # Sample every 10th
        continue
    if i > 200:  # Limit for speed
        break
    
    fees = snapshot['fee_rate'].dropna()
    time_stats.append({
        'timestamp': ts,
        'median': fees.median(),
        'mean': fees.mean(),
        'p25': fees.quantile(0.25),
        'p75': fees.quantile(0.75),
        'p95': fees.quantile(0.95),
        'tx_count': len(fees),
    })

time_df = pd.DataFrame(time_stats)
print(f'Collected stats for {len(time_df)} snapshots')

In [ ]:
# Plot fee rates over time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Fee rates
ax1 = axes[0]
ax1.fill_between(range(len(time_df)), time_df['p25'], time_df['p75'], alpha=0.3, label='25-75th percentile')
ax1.plot(time_df['median'], label='Median', linewidth=2)
ax1.plot(time_df['p95'], label='95th percentile', linestyle='--', alpha=0.7)
ax1.set_xlabel('Snapshot')
ax1.set_ylabel('Fee Rate (sat/vbyte)')
ax1.set_title('Fee Rates Over Time')
ax1.legend()

# Transaction count
ax2 = axes[1]
ax2.plot(time_df['tx_count'], color='green')
ax2.set_xlabel('Snapshot')
ax2.set_ylabel('Transaction Count')
ax2.set_title('Mempool Size Over Time')

plt.tight_layout()
plt.show()

## 4. Feature Correlations

How are the features related to each other?

In [ ]:
# Correlation matrix
corr = features[FEATURE_COLUMNS].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: fee_rate vs virtual_size (key relationship)
fig, ax = plt.subplots(figsize=(10, 6))

# Sample for plotting
sample = features.sample(min(5000, len(features)))
ax.scatter(sample['virtual_size'], sample['fee_rate'], alpha=0.3, s=10)
ax.set_xlabel('Virtual Size (vbytes)')
ax.set_ylabel('Fee Rate (sat/vbyte)')
ax.set_title('Fee Rate vs Transaction Size')
ax.set_xlim(0, sample['virtual_size'].quantile(0.99))
ax.set_ylim(0, sample['fee_rate'].quantile(0.99))
plt.show()

## 5. Stable vs Volatile Periods

Identify periods of different difficulty for prediction.

In [ ]:
# Calculate volatility (rolling std of median fee)
time_df['volatility'] = time_df['median'].rolling(window=5, min_periods=1).std()

# Classify periods
vol_threshold = time_df['volatility'].median()
time_df['period_type'] = np.where(time_df['volatility'] > vol_threshold, 'Volatile', 'Stable')

print(f'Volatility threshold: {vol_threshold:.2f}')
print(f'\nPeriod distribution:')
print(time_df['period_type'].value_counts())

In [ ]:
# Plot with period classification
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'Stable': 'green', 'Volatile': 'red'}
for period_type in ['Stable', 'Volatile']:
    mask = time_df['period_type'] == period_type
    ax.scatter(
        time_df[mask].index,
        time_df[mask]['median'],
        c=colors[period_type],
        label=period_type,
        alpha=0.6,
        s=20
    )

ax.set_xlabel('Snapshot')
ax.set_ylabel('Median Fee Rate (sat/vbyte)')
ax.set_title('Fee Rates by Period Volatility')
ax.legend()
plt.show()

In [ ]:
# Compare prediction difficulty in stable vs volatile periods
print('Prediction Difficulty Analysis:')
print('=' * 50)

for period_type in ['Stable', 'Volatile']:
    mask = time_df['period_type'] == period_type
    period_data = time_df[mask]
    
    # Naive prediction error: assume next = current
    naive_errors = np.abs(period_data['median'].diff().dropna())
    
    print(f'\n{period_type} Periods:')
    print(f'  Snapshots: {len(period_data)}')
    print(f'  Avg fee change: {naive_errors.mean():.2f} sat/vbyte')
    print(f'  Max fee change: {naive_errors.max():.2f} sat/vbyte')
    print(f'  => Harder to predict in volatile periods!')

## 6. 3h vs 1d Prediction Challenges

How does the prediction horizon affect difficulty?

In [ ]:
# Load a snapshot and check inclusion for different horizons
for ts, snapshot in iter_snapshots(dataset):
    # Calculate inclusion for both horizons
    included_3h = compute_inclusion_target(snapshot, '3h')
    included_1d = compute_inclusion_target(snapshot, '1d')
    
    print(f'Snapshot at {ts}')
    print(f'Transactions: {len(snapshot)}')
    print(f'\n3-hour horizon:')
    print(f'  Included: {included_3h.sum()} ({100*included_3h.mean():.1f}%)')
    print(f'\n1-day horizon:')
    print(f'  Included: {included_1d.sum()} ({100*included_1d.mean():.1f}%)')
    
    break

In [ ]:
# Collect inclusion rates over time for both horizons
horizon_stats = []
for i, (ts, snapshot) in enumerate(iter_snapshots(dataset)):
    if i % 20 != 0:  # Sample for speed
        continue
    if i > 200:
        break
    
    inc_3h = compute_inclusion_target(snapshot, '3h')
    inc_1d = compute_inclusion_target(snapshot, '1d')
    
    horizon_stats.append({
        'snapshot': i,
        'inclusion_3h': inc_3h.mean(),
        'inclusion_1d': inc_1d.mean(),
        'median_fee': snapshot['fee_rate'].median(),
    })

horizon_df = pd.DataFrame(horizon_stats)
print(f'Analyzed {len(horizon_df)} snapshots')

In [ ]:
# Plot inclusion rates by horizon
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Inclusion rates over time
ax1 = axes[0]
ax1.plot(horizon_df['inclusion_3h'] * 100, label='3-hour', linewidth=2)
ax1.plot(horizon_df['inclusion_1d'] * 100, label='1-day', linewidth=2)
ax1.set_xlabel('Snapshot')
ax1.set_ylabel('Inclusion Rate (%)')
ax1.set_title('Block Inclusion Rates by Prediction Horizon')
ax1.legend()
ax1.set_ylim(0, 100)

# Difference between horizons
ax2 = axes[1]
diff = (horizon_df['inclusion_1d'] - horizon_df['inclusion_3h']) * 100
ax2.bar(range(len(diff)), diff, alpha=0.7)
ax2.axhline(0, color='black', linestyle='-')
ax2.axhline(diff.mean(), color='red', linestyle='--', label=f'Mean: {diff.mean():.1f}%')
ax2.set_xlabel('Snapshot')
ax2.set_ylabel('1d - 3h Inclusion (%)')
ax2.set_title('Additional Inclusions from 3h to 1d Horizon')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'\nAverage additional inclusion from 3h to 1d: {diff.mean():.1f}%')

In [ ]:
# Key insight: What fee rate is needed for different horizons?
print('Fee Rate Requirements by Horizon:')
print('=' * 50)

for ts, snapshot in iter_snapshots(dataset):
    inc_3h = compute_inclusion_target(snapshot, '3h')
    inc_1d = compute_inclusion_target(snapshot, '1d')
    
    fees = snapshot['fee_rate']
    
    # What fee rate do you need to get included?
    included_3h_fees = fees[inc_3h]
    included_1d_fees = fees[inc_1d]
    not_included_fees = fees[~inc_1d]
    
    print(f'\nIncluded within 3h:')
    print(f'  Min fee rate: {included_3h_fees.min():.1f} sat/vbyte')
    print(f'  Median: {included_3h_fees.median():.1f} sat/vbyte')
    
    print(f'\nIncluded within 1d (but not 3h):')
    only_1d = fees[inc_1d & ~inc_3h]
    if len(only_1d) > 0:
        print(f'  Min fee rate: {only_1d.min():.1f} sat/vbyte')
        print(f'  Median: {only_1d.median():.1f} sat/vbyte')
    
    print(f'\nNot included within 1d:')
    if len(not_included_fees) > 0:
        print(f'  Max fee rate: {not_included_fees.max():.1f} sat/vbyte')
        print(f'  Median: {not_included_fees.median():.1f} sat/vbyte')
    
    break

## Summary

### Key Observations

1. **Fee Distribution:** Highly skewed with long tail of high fees
2. **Temporal Patterns:** Fees fluctuate significantly over time
3. **Volatility:** Some periods are stable, others highly volatile
4. **Horizon Difference:** 1d horizon has ~10-20% higher inclusion rate than 3h

### Implications for Predictors

- Use percentiles rather than mean (robust to outliers)
- Consider volatility as a feature
- Different strategies may be needed for 3h vs 1d
- Watch for regime changes in fee dynamics

In [ ]:
# Train/Val/Test split visualization
splits = get_train_val_test_splits(dataset)

print('Data Splits:')
print(f'  Train: {len(splits["train"])} snapshots')
print(f'  Val:   {len(splits["val"])} snapshots')
print(f'  Test:  {len(splits["test"])} snapshots')
print(f'\nRemember: Always evaluate on test set for final results!')